In [50]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
import prettytable as pt
import os
import warnings
warnings.filterwarnings("ignore")



###################################################################################################
                                                                                                  #
from rhoova.Client import *                                                                       #   
#Register and get api key from https://app.rhoova.com/ for ClientConfig("api key", "api secret")  #
config = ClientConfig("", "")                                                                     #
config = ClientConfig("iFpuqa43s0neL7NrF3fWb", "TYzuSHGUUe-B4UguM4k5wj4PhWSgZxwq")                # 
api = Api(config)                                                                                 # 
                                                                                                  #
###################################################################################################


directory = os.path.normpath(os.getcwd() + os.sep + os.pardir)
yielddatadirectory=directory+"/data/yielddata/bonddefiniton.csv"
marketdatadirectory=directory+"/data/marketdata/marketdata.csv"


bonddefiniton = pd.read_csv(yielddatadirectory,sep=';')
marketdata = pd.read_csv(marketdatadirectory,sep=';')

data=bonddefiniton.merge(marketdata[['valuationDate','isincode','value']], left_on='isincode', right_on='isincode')
data["valuationDate"]="2021-03-05"
data['issueDate'] = data['issueDate'].apply(lambda x: datetime.strptime(x, "%d.%m.%Y").strftime("%Y-%m-%d"))
data['maturityDate'] = data['maturityDate'].apply(lambda x: datetime.strptime(x, "%d.%m.%Y").strftime("%Y-%m-%d"))


In [92]:
bonddata ={
  "notional": 100,
  "settlementDate": "2021-03-09",
  "valuationDate": "2021-03-05",
  "buySell": "Buy",
  "fixedRateBondDefinition": {
    "businessDayConvention": "Unadjusted",
    "calendar": "Turkey",
    "coupon": 0.06625,
    "currency": "USD",
    "dateGeneration": "Backward",
    "dayCounter": "Thirty360E",
    "endOfMonth": True,
    "frequency": "Semiannual",
    "issueDate": "2014-02-19",
    "maturityDate": "2045-02-17",
    "maturityDayConvention": "Unadjusted",
    "redemption": 100
  },
  "discountCurve": {
    "calendar": "NullCalendar",
    "currency": "USD",
    "dayCounter": "Actual360",
    "instruments": {
      "BONDS": {
        "settlementDays": 2,
        "accuracy": 1e-5,
        "method": "NelsonSiegel",
        "numberOfIterations": 10000,
        "simplexLambda": 10  
      }
    },
    "intpMethod": "Linear",
    "period": "6M",
    "settlementDays": 2
  },
  "yieldData": data.to_dict('r')
}



In [93]:
try:
    resultdata = api.createTask(CalculationType.FIXED_RATE_BOND, bonddata,True)
    result=json.loads(resultdata["result"])
except RhoovaError as e:
    e.printPretty()    

In [94]:
npvTable = pt.PrettyTable(['Parameters', 'Value'])
npvTable.add_row(['PV', result.get('pv')])
npvTable.add_row(['Clean Price', result.get('cleanPrice')])
npvTable.add_row(['Dirty Price', result.get('dirtyPrice')])
npvTable.add_row(['Accrued Amount', result.get('accruedAmount')])
npvTable.add_row(['Yield to Maturity', 100*result.get('yieldToMaturity')])
npvTable.add_row(['Duration', result.get('duration')])
npvTable.add_row(['Modified Duration', result.get('modifiedDuration')])
npvTable.add_row(['Macualay Duration', result.get('macaulayDuration')])
npvTable.add_row(['Convexity', result.get('convexity')])
npvTable.add_row(['Bps', result.get('bps')])
npvTable.align = 'r'
npvTable.float_format = '.4'
print(npvTable)



+-------------------+----------+
|        Parameters |    Value |
+-------------------+----------+
|                PV |  96.5080 |
|       Clean Price |  96.1221 |
|       Dirty Price |  96.5270 |
|    Accrued Amount |   0.4049 |
| Yield to Maturity |   6.9595 |
|          Duration |  12.0452 |
| Modified Duration |  11.6402 |
| Macualay Duration |  12.0452 |
|         Convexity | 205.8660 |
|               Bps |   0.1164 |
+-------------------+----------+


In [95]:
cashflow=pd.DataFrame(result.get("data"))
cashflow[cashflow["termToMatByDay"]>0]

,fixingDate,accrualStart,accrualEnd,notional,currency,leg,payOrReceive,instrument,rate,zeroRate,spread,termToMatByDay,termToMatByYear,cashflow,discountFactor,cashflowPv
14,2021-02-15,2021-02-17,2021-08-17,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.022587,0,158,0.438889,3.312500,0.989887,3.279002
15,2021-08-13,2021-08-17,2022-02-17,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.027340,0,338,0.938889,3.312500,0.974361,3.227572
16,2022-02-15,2022-02-17,2022-08-17,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.031332,0,518,1.438889,3.312500,0.955586,3.165378
17,2022-08-15,2022-08-17,2023-02-17,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.035159,0,698,1.938889,3.312500,0.933738,3.093006
18,2023-02-15,2023-02-17,2023-08-17,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.038405,0,878,2.438889,3.312500,0.910198,3.015032
19,2023-08-15,2023-08-17,2024-02-19,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.041533,0,1060,2.944444,3.349306,0.884482,2.962400
20,2024-02-15,2024-02-19,2024-08-19,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.044219,0,1240,3.444444,3.312500,0.858300,2.843118
21,2024-08-15,2024-08-19,2025-02-17,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.046714,0,1418,3.938889,3.275694,0.831502,2.723746
22,2025-02-13,2025-02-17,2025-08-18,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.048880,0,1599,4.441667,3.330903,0.804405,2.679395
23,2025-08-14,2025-08-18,2026-02-17,100,USD,Fixed,Buy,Fixed Rate Bond,0.06625,0.050943,0,1778,4.938889,3.294097,0.777116,2.559895
